# Hipótese: `NR_TITULO_ELEITORAL_CANDIDATO` identifica a mesma pessoa entre eleições

Teste: pega títulos que aparecem em **2024** e em **2026**, busca em cada tabela e compara o nome
resultante. Se o título é chave estável da pessoa, o nome deve ser o mesmo (ou variar só em acento/grafia).

In [ ]:
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "dados").exists() else Path.cwd().parent
RAW = ROOT / "dados" / "raw" / "candidatos"

con = duckdb.connect()
_ = con.execute("INSTALL encodings; LOAD encodings;")
CSV_OPTS = "delim=';', quote='\"', header=true, encoding='cp1252', all_varchar=true, ignore_errors=true"

for ano in (2024, 2026):
    f = RAW / str(ano) / f"candidatos_{ano}" / f"consulta_cand_{ano}_BRASIL.csv"
    _ = con.execute(f"""
        CREATE TABLE cand_{ano} AS
        SELECT NR_TITULO_ELEITORAL_CANDIDATO AS titulo, NM_CANDIDATO AS nome, NM_URNA_CANDIDATO AS nome_urna,
               DT_NASCIMENTO AS nascimento, SG_UF AS uf, DS_CARGO AS cargo, SG_PARTIDO AS partido, SQ_CANDIDATO AS sq_candidato
        FROM read_csv('{f}', {CSV_OPTS})
        WHERE NR_TITULO_ELEITORAL_CANDIDATO NOT IN ('-4', '#NULO', '#NE', '')
    """)
    print(ano, con.execute(f"SELECT count(*), count(DISTINCT titulo) FROM cand_{ano}").fetchone())

2024 (463809, 462863)


2026 (20981, 20872)


## 1. Um título escolhido à mão

In [ ]:
# primeiro título que aparece nos dois anos (ordenado para ser reprodutível)
titulo = con.execute("""
    SELECT a.titulo FROM cand_2024 a JOIN cand_2026 b USING (titulo) ORDER BY a.titulo LIMIT 1
""").fetchone()[0]
print("título:", titulo)

for ano in (2024, 2026):
    print(f"\n--- {ano} ---")
    print(con.execute(f"SELECT nome, nome_urna, nascimento, uf, cargo, partido, sq_candidato FROM cand_{ano} WHERE titulo = ?", [titulo]).df().to_string(index=False))

NameError: name 'con' is not defined

## 2. Amostra aleatória de 10 títulos presentes nos dois anos

In [ ]:
con.execute("SELECT setseed(0.42)")
con.execute("""
    SELECT a.titulo, a.nome AS nome_2024, b.nome AS nome_2026, a.nascimento AS nasc_2024, b.nascimento AS nasc_2026,
           a.cargo AS cargo_2024, b.cargo AS cargo_2026
    FROM cand_2024 a JOIN cand_2026 b USING (titulo)
    ORDER BY random() LIMIT 10
""").df()

,titulo,nome_2024,nome_2026,nasc_2024,nasc_2026,cargo_2024,cargo_2026
0,011195431260,IVONETE ALMEIDA DE ANDRADE LUDGERIO,IVONETE ALMEIDA DE ANDRADE LUDGÉRIO,26/06/1966,26/06/1966,VEREADOR,1º SUPLENTE
1,065321700671,ROSY ANNE ALMODOVAS RODRIGUES RIBEIRO,ROSY ANNE ALMODOVAS RODRIGUES RIBEIRO,12/11/1979,12/11/1979,VEREADOR,DEPUTADO ESTADUAL
2,033165061384,RODINEI MOURA RAMOS,RODINEI MOURA RAMOS,28/10/1977,28/10/1977,VEREADOR,DEPUTADO ESTADUAL
3,002823492577,JIM DAVIS ROCHA DE ALMEIDA,JIM DAVIS ROCHA DE ALMEIDA,26/08/1978,26/08/1978,VEREADOR,DEPUTADO ESTADUAL
4,028044510973,HAMILTON MENDES DE OLIVEIRA,HAMILTON MENDES DE OLIVEIRA,24/09/1968,24/09/1968,VEREADOR,DEPUTADO FEDERAL
5,055149800744,FRANCISCA CERISLÂNDIA PIRES RODRIGUES DE SOUSA,FRANCISCA CERISLANDIA PIRES RODRIGUES DE SOUSA,14/09/1980,14/09/1980,VEREADOR,DEPUTADO FEDERAL
6,307488920167,JUCILEILA DOURADO CERQUEIRA,JUCILEILA DOURADO CERQUEIRA,28/08/1980,28/08/1980,VEREADOR,DEPUTADO FEDERAL
7,018277062135,SILVANEIDE RIBEIRO DE SOUZA,SILVANEIDE RIBEIRO DE SOUZA,24/05/1973,24/05/1973,VEREADOR,DEPUTADO FEDERAL
8,064454150507,JOSÉ ADEMAQUES DOS SANTOS,JOSÉ ADEMAQUES DOS SANTOS,05/07/1969,05/07/1969,VEREADOR,DEPUTADO ESTADUAL
9,012833682119,JOSÉ DE SOUZA SANTOS,JOSE DE SOUZA SANTOS,24/07/1966,24/07/1966,VEREADOR,DEPUTADO ESTADUAL


## 3. Em todos os títulos comuns: nome bate?

In [ ]:
con.execute("""
    WITH j AS (
        SELECT DISTINCT a.titulo, a.nome AS n24, b.nome AS n26, a.nascimento AS d24, b.nascimento AS d26
        FROM cand_2024 a JOIN cand_2026 b USING (titulo)
    )
    SELECT
        count(*)                                                        AS titulos_comuns,
        count(*) FILTER (WHERE n24 = n26)                               AS nome_igual,
        count(*) FILTER (WHERE n24 <> n26 AND strip_accents(upper(n24)) = strip_accents(upper(n26))) AS so_acento,
        count(*) FILTER (WHERE strip_accents(upper(n24)) <> strip_accents(upper(n26))) AS nome_diferente,
        count(*) FILTER (WHERE d24 <> d26)                              AS nascimento_diferente
    FROM j
""").df()

,titulos_comuns,nome_igual,so_acento,nome_diferente,nascimento_diferente
0,7972,6507,1227,238,39


### Os casos em que o nome difere (além de acento) — pessoa diferente ou mudança de nome?

In [ ]:
con.execute("""
    SELECT DISTINCT a.titulo, a.nome AS nome_2024, b.nome AS nome_2026, a.nascimento AS nasc_2024, b.nascimento AS nasc_2026
    FROM cand_2024 a JOIN cand_2026 b USING (titulo)
    WHERE strip_accents(upper(a.nome)) <> strip_accents(upper(b.nome))
    ORDER BY (a.nascimento <> b.nascimento) DESC, a.titulo
    LIMIT 30
""").df()

,titulo,nome_2024,nome_2026,nasc_2024,nasc_2026
0,000213111902,JURACI PEREIRA COSTA,JURACI PEREIRA DA COSTA,01/01/1967,01/10/1967
1,061934630710,ARIZA PEREIRA DA SILVA ALENCAR,ARIZA PEREIRA DA SILVA,18/08/1983,15/08/1983
2,000299512402,FRANCISCO EDUARDO MENEZES DE QUEIROZ,FRANCISCO EDUARDO MENESES DE QUEIROZ,13/12/1966,13/12/1966
3,000446100167,MÁRCIA LESENECHAL HISSETT,MARCIA LE SENECHAL HISSETT,18/05/1961,18/05/1961
4,001071852585,GIANCARLO DARLAN PINON NERY,GIANCARLO DARLA PINON NERY,03/12/1968,03/12/1968
5,001601012666,JOÃO GOMES DE SOUZA,JOÃO GOMES DE ARAÚJO,18/01/1960,18/01/1960
6,001648502402,"MARILETE VITORINO DE SIRQUEIRA ,",MARILETE VITORINO DE SIQUEIRA,21/03/1962,21/03/1962
7,002644692607,FÁBIA AUGUSTA DA SILVA GONÇALVES,FÁBIA AUGUSTA GONÇALVES MILITÃO,26/10/1975,26/10/1975
8,003788621082,NEILA MARIA GONÇALVES,NEILA MARIA GONCALVES MARTINS,21/05/1961,21/05/1961
9,004997011384,ODILEIDA MARIA DE SOUSA SAMPAIO,ODILEIDA MARIA SOUSA SAMPAIO,06/02/1955,06/02/1955
